In [ ]:
#grade (DO NOT DELETE THIS LINE)
import torch
import torch.nn as nn
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler

class SegmentedFashionMNIST(Dataset):
    def __init__(self, N, K, S=128):
        # N is number of images in dataset
        # K is number of FashionMNIST images to copy-paste into each example image
        # S is the square image size
        self.data = torchvision.datasets.FashionMNIST(root='./', download=True)
        self.N = N
        self.K = K
        self.S = S
        self.images, self.targets = self.generate_data()

    def generate_data(self):
        images = []
        targets = []
        for n in range(self.N):
            new_image = torch.zeros((1, self.S, self.S))
            new_label = torch.zeros((1, self.S, self.S))
            grid_size = self.S/np.sqrt(K)
            offset_coordinates = np.random.choice(np.arange(grid_size), size=(self.K, 2))
            random_coordinates = np.zeros((self.K, 2))
            n_cols = int(np.ceil(np.sqrt(K)))
            n_rows = int(np.ceil(K/n_cols))
            k = 0
            for k1 in range(n_rows):
                for k2 in range(n_cols):
                    random_coordinates[k] = np.array([np.clip(k1*grid_size+offset_coordinates[k, 0], 0, self.S-28),
                                                      np.clip(k2*grid_size+offset_coordinates[k, 1], 0, self.S-28)])
                    k += 1
                    if k >= self.K:
                        break
            random_coordinates = random_coordinates.astype(int)
            random_indices = np.random.choice(np.arange(len(self.data)), size=self.K)
            fashion_images = [self.data[random_indices[i]][0] for i in range(self.K)]
            fashion_labels = [self.data[random_indices[l]][1] for l in range(self.K)]
            for k in range(self.K):
                # threshold pixels to mask out objects in FashionMNIST images
                curr_image, curr_label = fashion_images[k], fashion_labels[k]
                curr_image = torchvision.transforms.functional.pil_to_tensor(curr_image)/255
                object_mask = (curr_image > (5/255)).float()
                curr_image *= object_mask
                curr_label_image = torch.ones_like(curr_image)*(curr_label+1)*object_mask
                tl_y, tl_x = random_coordinates[k]
                new_image[0, tl_y:tl_y+28, tl_x:tl_x+28] = new_image[0, tl_y:tl_y+28, tl_x:tl_x+28]*(1-object_mask)+curr_image
                new_label[0, tl_y:tl_y+28, tl_x:tl_x+28] = new_label[0, tl_y:tl_y+28, tl_x:tl_x+28]*(1-object_mask)+curr_label_image
            images.append(new_image)
            targets.append(new_label)
        return images, targets      
    
    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        return self.images[idx], self.targets[idx].long()

N = 100 # number of images
K = 5 # number of FashionMNIST objects placed on each image
dataset = SegmentedFashionMNIST(N, K)

In [ ]:
#grade (DO NOT DELETE THIS LINE)
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def iou(preds, targets):
    # preds is size (B, N_classes, H, W) for batch size B
    # targets is size (B, 1, H, W) for batch size B
    N_classes = preds.size(1) # number of items classes + 1 for background
    ious = np.zeros(N_classes) # per-class ious
    counts = np.zeros(N_classes) # number of images in batch with each class
    for b in range(preds.size(0)):
        values, locations = torch.max(preds[b], dim=0)
        # skip background class 0
        for c in range(1, N_classes):
            target_c = (targets[b]==c).float()
            if torch.sum(target_c).item():
                pred_c = (locations==c).float()
                intersection = torch.sum(pred_c*target_c)
                union = torch.sum((pred_c+target_c)>0)
                ious[c] += intersection/union # add to running IOU of class c
                counts[c] += 1 # increment count for class c
    return ious, counts

def training_loop(model, criterion, optimizer, n_epochs, train_loader, val_loader):
    loss_values, train_ious, val_ious = [], [], []
    for n in tqdm(range(n_epochs)):
        epoch_loss, epoch_ious, epoch_counts = 0, np.zeros(11), np.zeros(11)
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            B = x_batch.size(0)
            N_classes = 11
            # zero out gradients
            optimizer.zero_grad()
            # pass batch to model
            predictions = model(x_batch)
            # calculate loss
            loss = criterion(predictions.view(B, N_classes, -1), y_batch.view(B, -1))
            # backpropagate and update
            loss.backward() # backprop
            optimizer.step()
            # logging to update epoch_loss (add loss value) and epoch_iou (add current batch iou)
            epoch_loss += loss.item()
            ious, counts = iou(predictions, y_batch)
            epoch_ious += ious
            epoch_counts += counts
    
        loss_values.append(epoch_loss/len(train_loader))
        train_ious.append(epoch_ious[1:]/epoch_counts[1:])
        # validation performance
        epoch_val_ious, epoch_val_counts = np.zeros(11), np.zeros(11)
        for x_batch, y_batch in val_loader:
            # don't compute gradients since we are only evaluating the model
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            with torch.no_grad():
                # validation batch iou
                predictions = model(x_batch)
                ious, counts = iou(predictions, y_batch)
                epoch_val_ious += ious
                epoch_val_counts += counts
        val_ious.append(epoch_val_ious[1:]/epoch_val_counts[1:])
    return model, loss_values, train_ious, val_ious

In [ ]:
#grade (DO NOT DELETE THIS LINE)
class MyAutoEncoder(nn.Module):
    def __init__(self):
        # fill this is in as you like!
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1, 1), 
            nn.ReLU(), 
            nn.Conv2d(16, 16, 3, 1, 1), 
            nn.ReLU(), 
            nn.MaxPool2d(2, 2), 
            nn.Conv2d(16, 32, 3, 1, 1), 
            nn.ReLU(), 
            nn.MaxPool2d(2, 2), 
            nn.Conv2d(32, 64, 3, 1, 1), 
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, 2), 
            nn.Conv2d(32, 32, 3, 1, 1), 
            nn.ReLU(), 
            nn.ConvTranspose2d(32, 16, 2, 2), 
            nn.Conv2d(16, 16, 3, 1, 1), 
            nn.ReLU(), 
            nn.Conv2d(16, 11, 1)
        )
        
    def forward(self, x):
        # fill this in!
        # Note: do not apply sigmoid to your final model outputs since this is a multi-class problem with cross-entropy loss
        z = self.encoder(x)
        z = self.decoder(z)
        return z

In [ ]:
#grade (DO NOT DELETE THIS LINE)
# create dataset, do not change this!
N = 100
K = 5
N_train = 50
N_val = 50
dataset = SegmentedFashionMNIST(N, K)
indices = np.random.choice(np.arange(len(dataset)), size=N, replace=False)
np.random.shuffle(indices)
train_indices = indices[:N_train]
val_indices = indices[N_train:N_train+N_val]

# create dataloaders
batch_size = 4
train_loader = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_indices))
val_loader = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(val_indices))

# create model, fill this in as necessary for your model experimentation
model = MyAutoEncoder().to(device) # fill in constructor as necessary for your model

# criterion and optimizer
criterion = nn.CrossEntropyLoss().to(device) # cross-entropy loss since each pixel is evaluated by multi-class classification
lr = 1e-3 # tune this
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4) # tune this
n_epochs = 150 # tune this

# train
model, loss_values, train_ious, val_ious = training_loop(model, criterion, optimizer, n_epochs, train_loader, val_loader)
mean_train_ious = [np.mean(t) for t in train_ious]
mean_val_ious = [np.mean(v) for v in val_ious]

In [ ]:
# loss, training mIoU, validation mIoU
plt.figure(figsize=(15,6))
plt.subplot(131)
plt.semilogy(loss_values)
plt.grid(True)
plt.title('Loss values')
plt.xlabel('Epoch')
plt.subplot(132)
plt.plot(mean_train_ious)
# reference line for 0.8 validation mean IoU
plt.hlines([0.8], 0, len(mean_val_ious), colors='red', linestyles='dashed')
plt.ylim([0, 1])
plt.grid(True)
plt.title('Training IOUs')
plt.xlabel('Epoch')
plt.subplot(133)
plt.plot(mean_val_ious)
# reference line for 0.20 validation mean IoU
plt.hlines([0.20], 0, len(mean_val_ious), colors='red', linestyles='dashed')
plt.ylim([0, 1])
plt.grid(True)
plt.title('Validation IOUs')
plt.xlabel('Epoch')
plt.tight_layout()

# per class IoUs, not necessary for homework, but provided for those who are interested
plt.figure(figsize=(20, 15))
for i in range(10):
    plt.subplot(3, 4, i+1)
    plt.plot([t[i] for t in train_ious], label='Train')
    plt.plot([v[i] for v in val_ious], label='Validation')
    plt.ylim([0, 1])
    plt.legend()
    plt.title('Class {} IOUs'.format(i+1))
plt.tight_layout()

In [ ]:
# Part (c): visualize model output
idx = 1 # try changing this
train_idx = train_indices[idx]
val_idx = val_indices[idx]

# training image
image, target = dataset[train_idx]
with torch.no_grad():
    # add batch dimension then remove batch dimension after passing to model
    prediction = model(image.unsqueeze(0).to(device)).squeeze(0) 
    _, prediction_image = torch.max(prediction, dim=0)
    prediction_image = prediction_image.cpu() # move to cpu for visualization

plt.figure(figsize=(15, 6))
plt.subplot(131)
plt.imshow(image.squeeze(0).numpy(), 'gray') # remove color channel dimension with .squeeze(0)
plt.axis(False)
plt.title('Training Input Image')
plt.subplot(132)
plt.imshow(target.squeeze(0).numpy(), 'tab20', interpolation='nearest')
plt.axis(False)
plt.title('Ground Truth')
plt.subplot(133)
plt.imshow(prediction_image.numpy(), 'tab20', interpolation='nearest')
plt.axis(False)
plt.title('Model Prediction')
plt.tight_layout()

# validation image
image, target = dataset[val_idx]
with torch.no_grad():
    # add batch dimension then remove batch dimension after passing to model
    prediction = model(image.unsqueeze(0).to(device)).squeeze(0) 
    _, prediction_image = torch.max(prediction, dim=0)
    prediction_image = prediction_image.cpu() # move to cpu for visualization

plt.figure(figsize=(15, 6))
plt.subplot(131)
plt.imshow(image.squeeze(0).numpy(), 'gray') # remove color channel dimension with .squeeze(0)
plt.axis(False)
plt.title('Validation Input Image')
plt.subplot(132)
plt.imshow(target.squeeze(0).numpy(), 'tab20', interpolation='nearest')
plt.axis(False)
plt.title('Ground Truth')
plt.subplot(133)
plt.imshow(prediction_image.numpy(), 'tab20', interpolation='nearest')
plt.axis(False)
plt.title('Model Prediction')
plt.tight_layout()